# Postprocess – unsteady mixer experiments

Reads the optimisation-history CSV for each `.case` experiment and generates
comparative convergence plots of:
* the **total** objective,
* every **partial** objective individually, and
* every **constraint**.

For **unsteady** cases (`"unsteady": true`) objectives/constraints are time integrals;
they are divided by `end_time` before plotting so they compare directly with steady values.

The CSV reader silently drops repeated header rows (cluster restarts).

Edit the *Configuration* cell below to point `DATA_PATH` at your data root.

In [ ]:
# ── Configuration ───────────────────────────────────────────────
DATA_PATH = "/mnt/o/Public/Project-111856/Simulations/unsteady_mixer"
CASES_DIR = "."              # directory containing the .case files (this notebook's dir)
OUTDIR    = "results/plots"   # where PNGs are saved (relative to CASES_DIR)

In [ ]:
%matplotlib inline
import csv
import glob
import json
import os
import re
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.4,
    "figure.figsize": (8, 4.5),
})

In [ ]:
# ── Helper functions ──────────────────────────────────────────────

def _strip_comments(text: str) -> str:
    """Remove // line-comments so JSON5-style .case files parse cleanly."""
    return re.sub(r"//.*?$", "", text, flags=re.MULTILINE)


def load_case(path: str) -> dict:
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return json.loads(_strip_comments(raw))


def find_case_files(cases_dir: str) -> List[str]:
    return sorted(glob.glob(os.path.join(cases_dir, "*.case")))


def is_unsteady(case_obj: dict) -> bool:
    if case_obj.get("unsteady", False):
        return True
    inner = case_obj.get("case", {})
    return bool(isinstance(inner, dict) and inner.get("unsteady", False))


def get_end_time(case_obj: dict) -> Optional[float]:
    """Return end_time from case JSON (tries common locations/spellings)."""
    for key in ("end_time", "endtime", "endTime", "end"):
        t = case_obj.get("case", {}).get("time", {})
        if isinstance(t, dict) and key in t:
            return float(t[key])
        if key in case_obj:
            return float(case_obj[key])
    return None


def find_data_dir(data_path: str, case_path: str) -> Optional[str]:
    """Locate the result sub-folder for *case_path* under *data_path*."""
    stem = os.path.splitext(os.path.basename(case_path))[0]
    for entry in glob.glob(os.path.join(data_path, stem + "*")):
        if os.path.isdir(entry):
            return entry
    for root, dirs, _ in os.walk(data_path):
        for d in dirs:
            if stem in d:
                return os.path.join(root, d)
    return None


def find_history_csv(data_dir: str) -> Optional[str]:
    """Return the optimisation-history CSV in *data_dir*, preferring optimization_data.csv."""
    all_csvs = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)
    if not all_csvs:
        return None
    named = [p for p in all_csvs if os.path.basename(p) == "optimization_data.csv"]
    if named:
        return named[0]
    prioritised = [
        p for p in all_csvs
        if re.search(r"history|opt|objective|iter", os.path.basename(p), re.I)
    ]
    candidates = prioritised or all_csvs
    candidates.sort(key=os.path.getsize, reverse=True)
    return candidates[0]


def read_history(path: str) -> pd.DataFrame:
    """
    Read a CSV optimisation history, silently skipping repeated header rows
    that appear whenever the run was restarted on the cluster.
    """
    with open(path, encoding="utf-8", errors="replace") as f:
        lines = [ln for ln in f.readlines() if ln.strip() and not ln.startswith("#")]
    if not lines:
        return pd.DataFrame()
    try:
        delim = csv.Sniffer().sniff("".join(lines[:20])).delimiter
    except csv.Error:
        delim = ","
    header_line = lines[0].strip()
    columns = [c.strip() for c in header_line.split(delim)]
    rows = []
    for ln in lines[1:]:
        if ln.strip() == header_line:  # repeated header from cluster restart
            continue
        try:
            row = next(csv.reader([ln], delimiter=delim))
        except StopIteration:
            continue
        row = (row + [""] * len(columns))[: len(columns)]
        rows.append(row)
    df = pd.DataFrame(rows, columns=columns)
    df.columns = df.columns.str.strip()
    for col in df.columns:
        s = df[col].astype(str).str.strip()
        df[col] = pd.to_numeric(s.mask(s == ""), errors="coerce")
    return df


def load_experiment(case_path: str, data_path: str) -> Optional[dict]:
    """
    Load one experiment.  Returns a dict with all data needed for plotting,
    or None if the result folder / CSV cannot be found.
    """
    case = load_case(case_path)
    unsteady = is_unsteady(case)
    end_time = get_end_time(case)

    data_dir = find_data_dir(data_path, case_path)
    if data_dir is None:
        print(f"[skip] no data folder found for {os.path.basename(case_path)}")
        return None

    csv_path = find_history_csv(data_dir)
    if csv_path is None:
        print(f"[skip] no history CSV in {data_dir}")
        return None

    df = read_history(csv_path)
    if df.empty:
        print(f"[skip] empty CSV for {os.path.basename(case_path)}")
        return None

    # Choose iteration-index column
    idx_col = next(
        (c for c in df.columns if c.lower() in {"iter", "iteration", "it", "step", "n", "k"}),
        None,
    )
    if idx_col is None:
        df.insert(0, "iter", np.arange(len(df)))
        idx_col = "iter"
    else:
        df = df.sort_values(idx_col).reset_index(drop=True)

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    skip = {idx_col, "time", "t", "timestamp"}
    metric_cols = [c for c in numeric_cols if c not in skip]

    # For unsteady cases divide time-integral values by end_time
    if unsteady:
        if end_time:
            df[metric_cols] = df[metric_cols] / end_time
        else:
            print(f"[warn] unsteady {os.path.basename(case_path)} has no end_time – not averaged")

    # Classify columns: exclude weights, KKT, scaling, backend/subsolver auxiliary cols
    _aux = re.compile(r"\.weight$|^KKT|^scaling|^backend:|^subsolver:", re.I)
    total_col = next((c for c in metric_cols if "total" in c.lower()), None)
    obj_cols = [
        c for c in metric_cols
        if c != total_col and not _aux.search(c)
        and not any(k in c.lower() for k in ["g_", "constraint", "constr", "limit"])
    ]
    constr_cols = [
        c for c in metric_cols
        if any(k in c.lower() for k in ["g_", "constraint", "constr", "limit"])
    ]

    return dict(
        label=os.path.splitext(os.path.basename(case_path))[0],
        case_path=case_path,
        df=df,
        idx_col=idx_col,
        metric_cols=metric_cols,
        total_col=total_col,
        obj_cols=obj_cols,
        constr_cols=constr_cols,
        unsteady=unsteady,
        end_time=end_time,
    )

In [ ]:
# ── Discover and load all experiments ──────────────────────────────────
case_files = find_case_files(CASES_DIR)
print(f"Found {len(case_files)} case file(s) in '{CASES_DIR}':")
for cf in case_files:
    print(" ", os.path.basename(cf))
print()

experiments = []
for cf in case_files:
    exp = load_experiment(cf, DATA_PATH)
    if exp is not None:
        u_tag  = "unsteady" if exp["unsteady"] else "steady"
        et_tag = f", end_time={exp['end_time']}" if exp["unsteady"] and exp["end_time"] else ""
        print(f"  Loaded {exp['label']:40s} ({u_tag}{et_tag})  →  {len(exp['df'])} iterations")
        experiments.append(exp)

if not experiments:
    print("\nNo experiments loaded – check DATA_PATH.")

In [ ]:
# ── Total objective (comparative) ──────────────────────────────────────────
os.makedirs(OUTDIR, exist_ok=True)

fig, ax = plt.subplots()
for exp in experiments:
    df, idx = exp["df"], exp["idx_col"]
    tc = exp["total_col"]
    y = df[tc] if tc else df[exp["obj_cols"] or exp["metric_cols"]].sum(axis=1)
    ax.plot(df[idx], y, label=exp["label"])

ax.set_xlabel("Iteration")
ax.set_ylabel("Total objective")
ax.set_title("Optimisation history – total objective")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "total_objective.png"))
plt.show()

In [ ]:
# ── Objective components per case (all components on one plot per experiment) ──
obj_keys = sorted({c for exp in experiments for c in exp["obj_cols"]})

if not obj_keys:
    print("No partial-objective columns detected.")
else:
    for exp in experiments:
        df, idx = exp["df"], exp["idx_col"]
        cols = [c for c in obj_keys if c in df.columns]
        if not cols:
            continue
        fig, ax = plt.subplots()
        for col in cols:
            ax.plot(df[idx], df[col], label=col)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Objective component value")
        ax.set_title(f"Objective components – {exp['label']}")
        ax.legend()
        fig.tight_layout()
        safe = re.sub(r"[^\w-]", "_", exp["label"])
        fig.savefig(os.path.join(OUTDIR, f"obj_components_{safe}.png"))
        plt.show()

In [ ]:
# ── Objective components – comparative (one plot per component, all cases) ──
obj_keys = sorted({c for exp in experiments for c in exp["obj_cols"]})

if not obj_keys:
    print("No partial-objective columns detected.")
else:
    for key in obj_keys:
        fig, ax = plt.subplots()
        for exp in experiments:
            if key not in exp["df"].columns:
                continue
            ax.plot(exp["df"][exp["idx_col"]], exp["df"][key], label=exp["label"])
        ax.set_xlabel("Iteration")
        ax.set_ylabel(key)
        ax.set_title(f"Objective component – {key}")
        ax.legend()
        fig.tight_layout()
        safe = re.sub(r"[^\w-]", "_", key)
        fig.savefig(os.path.join(OUTDIR, f"objective_{safe}.png"))
        plt.show()

In [ ]:
# ── Constraints (one plot each) ─────────────────────────────────────────────
constr_keys = sorted({c for exp in experiments for c in exp["constr_cols"]})

if not constr_keys:
    print("No constraint columns detected.")
else:
    for key in constr_keys:
        fig, ax = plt.subplots()
        for exp in experiments:
            if key not in exp["df"].columns:
                continue
            ax.plot(exp["df"][exp["idx_col"]], exp["df"][key], label=exp["label"])
        ax.set_xlabel("Iteration")
        ax.set_ylabel(key)
        ax.set_title(f"Constraint – {key}")
        ax.legend()
        fig.tight_layout()
        safe = re.sub(r"[^\w-]", "_", key)
        fig.savefig(os.path.join(OUTDIR, f"constraint_{safe}.png"))
        plt.show()